# 장기 기억: Preference

`add_preference()`로 사용자 선호도를 저장하고, 사용자와 선호도의 연결을 Neo4j 그래프로 확인한다.
선호도는 `Preference` 노드로 저장되며, `user_identifier`를 지정하면 `User` 노드와 `HAS_PREFERENCE` 관계로 연결된다.

실행 흐름은 클라이언트 연결, 선호도 저장, 시각화, 의미 기반 검색, 데이터 삭제, 연결 종료 순서이다.
[엔티티와 관계 실험](02_long_term_memory_entity.ipynb), [Fact 실험](03_long_term_memory_fact.ipynb)과 독립적으로 실행할 수 있다.

## 실행 준비

로그 출력을 설정하고 공통 설정 모듈에서 `MemorySettings`를 구성한다.
클라이언트를 한 번 연결한 뒤 저장·시각화·검색·삭제 단계에서 재사용한다. 각 노트북은 별도의 커널에서 독립적으로 실행할 수 있다.

In [ ]:
import logging

logging.getLogger("neo4j.notifications").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)

In [ ]:
from config.memory_client import load_memory_settings, print_memory_settings
from neo4j_agent_memory import MemorySettings

settings: MemorySettings = load_memory_settings()
print_memory_settings(settings)

In [ ]:
from neo4j_agent_memory import MemoryClient

memory_client = MemoryClient(settings)

try:
    await memory_client.connect()
except:
    await memory_client.close()
    raise

print("✓ Memory client opened")

## 사용자 식별자 정의

`user_id`는 선호도를 연결할 `User.identifier` 값이다. 대화의 `session_id`와는 별개이며, 이 실험은 대화를 생성하지 않는다.
저장·시각화·삭제 단계에서 같은 식별자를 사용한다. 식별자를 변경해도 기존 데이터는 삭제되지 않는다.

In [ ]:
user_id = "04_long_term_memory_preference"

## 사용자 선호도 저장

브랜드·음식·취미 선호도를 저장하고, 반환된 객체를 `stored_preferences`에 보관한다.
중복 제거가 활성화된 경우 같은 범주의 유사한 기존 선호도가 재사용될 수 있으므로, 시각화와 삭제에는 반환 객체의 ID를 사용한다.
기존 선호도가 재사용되더라도 현재 사용자의 `HAS_PREFERENCE` 연결은 생성된다.

저장 셀을 다시 실행하면 `stored_preferences`는 마지막 실행의 반환 객체로 갱신된다.
중복 제거 여부에 따라 이전 실행의 별도 노드가 남을 수 있으므로 반복 실험 전에는 정리 셀을 실행한다.

In [ ]:
preferences = [
    ("brand", "Carhartt 제품을 선호함"),
    ("food", "맑은 국물을 선호함"),
    ("hobby", "온라인 게임을 선호함"),
]

stored_preferences = []

for category, preference in preferences:
    stored_preference = await memory_client.long_term.add_preference(
        category=category,
        preference=preference,
        user_identifier=user_id,
        context="User stated preference",
        generate_embedding=True,
    )
    stored_preferences.append(stored_preference)
    print(f"✓ [{stored_preference.category}] {stored_preference.preference} ({stored_preference.id})")

## 사용자와 선호도 시각화

현재 `user_id`와 `stored_preferences`의 ID를 기준으로 `User → HAS_PREFERENCE → Preference` 경로를 조회한다.
공유 모듈 `graph_visualizer`의 `visualize()`에 기존 세션과 쿼리 파라미터를 전달하여 그래프를 표시한다.
노드 속성에서 사용자 식별자, 선호도 범주·내용·맥락·신뢰도를 확인한다.

조회 대상은 이번 저장 셀에서 반환된 선호도와 현재 사용자의 연결이다. 같은 선호도에 연결된 다른 사용자는 표시하지 않는다.
세션은 조회 후 닫히고 메모리 클라이언트는 이후 실험에서 재사용한다.

In [ ]:
from typing import LiteralString

from graph_visualizer import visualize

graph_query: LiteralString = """
MATCH path = (u:User {identifier: $user_id})-[:HAS_PREFERENCE]->(p:Preference)
WHERE p.id IN $preference_ids
RETURN path
"""

async with memory_client.graph.session() as graph_session:
    await visualize(
        graph_session,
        graph_query,
        {
            "user_id": user_id,
            "preference_ids": [str(pref.id) for pref in stored_preferences],
        },
    )

## 질의 기반 선호도 검색

`search_preferences()`로 검색어와 의미적으로 유사한 선호도를 찾고 유사도를 출력한다.
각 검색어의 반환 개수와 상위 2개 항목을 확인한다.

현재 SDK의 이 메서드는 사용자 식별자 필터를 제공하지 않으므로, 다른 사용자의 선호도도 검색 결과에 포함될 수 있다.
앞선 사용자별 그래프 조회와 검색 범위를 구별한다. 검색 결과 `prefs`는 삭제 대상에 사용하지 않는다.

In [ ]:
queries = ["Carhartt", "국물"]

for query in queries:
    prefs = await memory_client.long_term.search_preferences(
        query=query,
        limit=3
    )
    print(f"  Query '{query}': {len(prefs)}개 발견")
    for pref in prefs[:2]:
        similarity = pref.metadata.get("similarity", 0)
        print(f"    - [{pref.category}] {pref.preference} (유사도: {similarity:.3f})")


## 실험 데이터 삭제

현재 `user_id`와 `stored_preferences`의 ID에 해당하는 `HAS_PREFERENCE` 관계를 삭제한다.
이후 대상 선호도와 사용자 중 다른 연결이 없는 노드만 삭제한다. 다른 사용자나 엔티티와 연결된 선호도는 보존한다.
삭제한 관계·선호도·사용자 수와 다른 연결 때문에 보존된 선호도 수를 출력한다.

삭제는 하나의 쿼리로 처리되어 오류 발생 시 함께 롤백된다. 중복 제거로 기존 선호도가 재사용된 경우에도 같은 ID가 정리 대상이 된다.
데이터를 보존하는 경우 이 셀을 생략하고 클라이언트 종료 셀만 실행한다.

In [ ]:
from typing import LiteralString

cleanup_query: LiteralString = """
CALL () {
    MATCH (:User {identifier: $user_id})-[r:HAS_PREFERENCE]->(p:Preference)
    WHERE p.id IN $preference_ids
    DELETE r
    RETURN count(*) AS deleted_relationships
}
CALL () {
    MATCH (p:Preference)
    WHERE p.id IN $preference_ids AND NOT (p)--()
    DELETE p
    RETURN count(*) AS deleted_preferences
}
CALL () {
    MATCH (u:User {identifier: $user_id})
    WHERE NOT (u)--()
    DELETE u
    RETURN count(*) AS deleted_users
}
CALL () {
    MATCH (p:Preference)
    WHERE p.id IN $preference_ids
    RETURN count(*) AS retained_preferences
}
RETURN deleted_relationships, deleted_preferences, deleted_users, retained_preferences
"""

async with memory_client.graph.session() as graph_session:
    result = await graph_session.run(
        cleanup_query,
        {
            "user_id": user_id,
            "preference_ids": [str(pref.id) for pref in stored_preferences],
        },
    )
    deleted = await result.single(strict=True)
    await result.consume()

print(f"✓ Deleted preference relationships: {deleted['deleted_relationships']}")
print(f"✓ Deleted preferences: {deleted['deleted_preferences']}")
print(f"✓ Deleted users: {deleted['deleted_users']}")
print(f"  Retained preferences with other connections: {deleted['retained_preferences']}")

## 메모리 클라이언트 종료

`close()`로 메모리 클라이언트의 연결과 백그라운드 작업을 정리한다.
연결 종료는 DB에 저장된 데이터를 삭제하지 않으므로, 데이터 삭제 여부와 관계없이 실험을 마칠 때 실행한다.
종료 후 실험을 재개하려면 클라이언트 생성 및 연결 셀부터 다시 실행한다.

In [ ]:
await memory_client.close()
print("✓ Memory client closed")